**Public archival copy.** Saved cell outputs (including embedded images) and Colab-specific runtime metadata have been removed. Original code and explanatory cell content remain unchanged. Some code refers to Google Drive paths and intermediate research artifacts that are not distributed here. The Fitzpatrick17k image files and the researchers' individual access link are **not included**. See `notebooks/README.md` for access and reproducibility notes.


In [ ]:
# ============================================
# Section 1.0
# Mount Google Drive
# ============================================

from google.colab import drive

drive.mount(
    "/content/drive"
)

In [ ]:
# ============================================
# Section 1.1
# Phase 8 Imports and Paths
# ============================================

from pathlib import Path

import json
import random
import math
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader
)

from torchvision import (
    models,
    transforms
)

from torchvision.models import (
    ResNet50_Weights
)

from PIL import Image

import matplotlib.pyplot as plt


ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)


DATA_PATH = (
    ARTIFACT_PATH
    / "dataset_splits"
)


PHASE6_PATH = (
    ARTIFACT_PATH
    / "Phase_6_Domain_Adaptation"
)


PHASE7_PATH = (
    ARTIFACT_PATH
    / "Phase_7_Broader_Fairness_Analysis"
)


PHASE8_PATH = (
    ARTIFACT_PATH
    / "Phase_8_GradCAM_Explainability"
)


PHASE8_CONFIG_PATH = (
    PHASE8_PATH
    / "config"
)


PHASE8_TABLE_PATH = (
    PHASE8_PATH
    / "tables"
)


PHASE8_FIGURE_PATH = (
    PHASE8_PATH
    / "figures"
)


PHASE8_HEATMAP_PATH = (
    PHASE8_PATH
    / "heatmaps"
)


PHASE8_MANIFEST_PATH = (
    PHASE8_PATH
    / "manifests"
)


PHASE8_AUDIT_PATH = (
    PHASE8_PATH
    / "audits"
)


for directory in [

    PHASE8_PATH,
    PHASE8_CONFIG_PATH,
    PHASE8_TABLE_PATH,
    PHASE8_FIGURE_PATH,
    PHASE8_HEATMAP_PATH,
    PHASE8_MANIFEST_PATH,
    PHASE8_AUDIT_PATH

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


GLOBAL_SEED = 42

random.seed(
    GLOBAL_SEED
)

np.random.seed(
    GLOBAL_SEED
)

torch.manual_seed(
    GLOBAL_SEED
)


DEVICE = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Device:",
    DEVICE
)

print(
    "Phase 8 root:",
    PHASE8_PATH
)

print(
    "✓ Phase 8 directories ready."
)

In [ ]:
# ============================================
# Section 2.1
# Verify Frozen Phase 6 and Phase 7
# ============================================

PHASE6_COMPLETION_FILE = (

    PHASE6_PATH
    / "manifests"
    / "Phase6_Completion_Manifest.json"
)


PHASE7_COMPLETION_FILE = (

    PHASE7_PATH
    / "manifests"
    / "Phase7_Completion_Manifest.json"
)


assert PHASE6_COMPLETION_FILE.exists()
assert PHASE7_COMPLETION_FILE.exists()


with open(
    PHASE6_COMPLETION_FILE,
    "r"
) as f:

    phase6_manifest = json.load(
        f
    )


with open(
    PHASE7_COMPLETION_FILE,
    "r"
) as f:

    phase7_manifest = json.load(
        f
    )


assert (
    phase6_manifest[
        "Status"
    ]
    ==
    "COMPLETE"
)


assert (
    phase6_manifest[
        "Frozen"
    ]
    is True
)


assert (
    phase7_manifest[
        "Status"
    ]
    ==
    "COMPLETE"
)


assert (
    phase7_manifest[
        "Frozen"
    ]
    is True
)


assert (
    float(
        phase6_manifest[
            "Selected_Lambda"
        ]
    )
    ==
    0.0
)


PHASE8_SELECTED_LAMBDA = 0.0


PHASE8_SEEDS = [
    42,
    43,
    44,
    45,
    46
]


print(
    "✓ Phase 6 COMPLETE and frozen."
)

print(
    "✓ Phase 7 COMPLETE and frozen."
)

print(
    "✓ Phase 8 authoritative lambda:",
    PHASE8_SELECTED_LAMBDA
)

print(
    "✓ Phase 8 model seeds:",
    PHASE8_SEEDS
)

In [ ]:
# ============================================
# Section 3.1
# Restore Frozen Image Preprocessing
# ============================================

PREPROCESSING_CONFIG_FILE = (

    ARTIFACT_PATH
    / "preprocessing_config.json"
)


assert (
    PREPROCESSING_CONFIG_FILE.exists()
)


with open(
    PREPROCESSING_CONFIG_FILE,
    "r"
) as f:

    preprocessing_config = json.load(
        f
    )


IMAGE_SIZE = int(
    preprocessing_config[
        "image_size"
    ]
)


NORMALIZE_MEAN = (
    preprocessing_config[
        "normalization"
    ][
        "mean"
    ]
)


NORMALIZE_STD = (
    preprocessing_config[
        "normalization"
    ][
        "std"
    ]
)


eval_transform = transforms.Compose([

    transforms.Resize(
        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=NORMALIZE_MEAN,
        std=NORMALIZE_STD
    )
])


print(
    "Image size:",
    IMAGE_SIZE
)

print(
    "Mean:",
    NORMALIZE_MEAN
)

print(
    "Std:",
    NORMALIZE_STD
)

print(
    "✓ Frozen evaluation preprocessing restored."
)

In [ ]:
# ============================================
# Section 4.1
# Load Fixed Test Metadata
# and Frozen Lambda=0 Predictions
# ============================================

FAIR_TEST_FILE = (
    DATA_PATH
    / "fair_test.csv"
)


DARK_TEST_FILE = (
    DATA_PATH
    / "dark_test.csv"
)


PHASE6_PREDICTION_FILE = (

    PHASE6_PATH
    / "predictions"
    / "Phase6_Final_Test_Predictions.csv"
)


assert FAIR_TEST_FILE.exists()
assert DARK_TEST_FILE.exists()
assert PHASE6_PREDICTION_FILE.exists()


fair_test_df = pd.read_csv(
    FAIR_TEST_FILE
)


dark_test_df = pd.read_csv(
    DARK_TEST_FILE
)


fair_test_df[
    "Evaluation_Domain"
] = "Fair"


dark_test_df[
    "Evaluation_Domain"
] = "Dark"


test_metadata_df = pd.concat(

    [
        fair_test_df,
        dark_test_df
    ],

    ignore_index=True
)


phase6_predictions_df = pd.read_csv(
    PHASE6_PREDICTION_FILE
)


phase8_predictions_df = (

    phase6_predictions_df[

        phase6_predictions_df[
            "Lambda"
        ]
        ==
        0.0
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


print(
    "Test images:",
    len(
        test_metadata_df
    )
)

print(
    "Selected prediction rows:",
    len(
        phase8_predictions_df
    )
)


assert (
    len(
        test_metadata_df
    )
    ==
    649
)


assert (
    len(
        phase8_predictions_df
    )
    ==
    649 * 5
)


print(
    "✓ Phase 8 source data loaded."
)

In [ ]:
# ============================================
# Section 4.2
# Build Five-Seed Ensemble Prediction Table
# ============================================

prediction_audit = (

    phase8_predictions_df

    .groupby(
        "md5hash"
    )[
        "Seed"
    ]

    .nunique()
)


assert (
    prediction_audit.min()
    ==
    5
)


assert (
    prediction_audit.max()
    ==
    5
)


ensemble_df = (

    phase8_predictions_df

    .groupby(
        "md5hash",
        as_index=False
    )

    .agg(

        Ensemble_Probability=(
            "probability",
            "mean"
        ),

        Probability_SD=(
            "probability",
            "std"
        )
    )
)


ensemble_df[
    "Ensemble_Prediction"
] = (

    ensemble_df[
        "Ensemble_Probability"
    ]
    >=
    0.5

).astype(int)


eps = 1e-12


p = np.clip(

    ensemble_df[
        "Ensemble_Probability"
    ]
    .to_numpy(
        dtype=float
    ),

    eps,

    1.0 - eps
)


ensemble_df[
    "Ensemble_Entropy"
] = -(

    p
    *
    np.log(
        p
    )

    +

    (
        1.0
        -
        p
    )
    *
    np.log(
        1.0
        -
        p
    )
)


metadata_columns = [

    "md5hash",

    "image_path",

    "target",

    "skin_tone",

    "Evaluation_Domain",

    "fitzpatrick_scale",

    "fitzpatrick_centaur",

    "label"
]


available_metadata_columns = [

    column

    for column in metadata_columns

    if column
    in test_metadata_df.columns
]


phase8_case_df = ensemble_df.merge(

    test_metadata_df[
        available_metadata_columns
    ]
    .drop_duplicates(
        subset=[
            "md5hash"
        ]
    ),

    on="md5hash",

    how="left",

    validate="one_to_one"
)


phase8_case_df[
    "Correct"
] = (

    phase8_case_df[
        "Ensemble_Prediction"
    ]
    ==
    phase8_case_df[
        "target"
    ]
)


print(
    "Image-level cases:",
    len(
        phase8_case_df
    )
)


print(
    "\nCorrectness:"
)


print(

    phase8_case_df[
        "Correct"
    ]
    .value_counts()
)


print(
    "\nEntropy summary:"
)


print(

    phase8_case_df[
        "Ensemble_Entropy"
    ]
    .describe()
)


assert (
    len(
        phase8_case_df
    )
    ==
    649
)


print(
    "\n✓ Five-seed ensemble table created."
)

In [ ]:
# ============================================
# Section 5.1
# Freeze Phase 8 XAI Analysis Plan
# ============================================

phase8_xai_plan = {

    "Phase":
        "Phase 8",

    "Title":
        "Grad-CAM++ Explainability Analysis",

    "Authoritative_Model":
        "Phase 6 selected lambda=0",

    "Model_Seeds":
        [
            42,
            43,
            44,
            45,
            46
        ],

    "Explanation_Method":
        "Grad-CAM++",

    "Target_Layer":
        (
            "ResNet50 final convolutional "
            "feature layer before global average pooling"
        ),

    "Seed_Aggregation":
        (
            "Generate one normalized Grad-CAM++ map "
            "per seed and average across five seeds"
        ),

    "Case_Selection_Uses_Heatmaps":
        False,

    "Primary_Case_Dimensions":
        [
            "Fair versus Dark",
            "Benign versus Malignant",
            "Correct versus Incorrect",
            "Low versus High predictive entropy"
        ],

    "Phase7_Targeted_Analysis":
        (
            "Fitzpatrick VI high-entropy cases examined "
            "because Phase 7 found a Holm-supported "
            "omnibus Fitzpatrick entropy difference."
        ),

    "Heatmaps_Used_For_Model_Selection":
        False,

    "Model_Selection_Reopened":
        False,

    "Frozen_Before_Heatmap_Inspection":
        True
}


PHASE8_XAI_PLAN_FILE = (

    PHASE8_CONFIG_PATH
    / "Phase8_XAI_Analysis_Plan.json"
)


with open(
    PHASE8_XAI_PLAN_FILE,
    "w"
) as f:

    json.dump(
        phase8_xai_plan,
        f,
        indent=4
    )


print(
    "✓ Phase 8 XAI plan frozen before heatmap generation."
)

print(
    PHASE8_XAI_PLAN_FILE
)

In [ ]:
# ============================================
# Section 5.2
# Audit XAI Case Availability
# ============================================

case_availability_df = (

    phase8_case_df

    .groupby(

        [
            "Evaluation_Domain",
            "target",
            "Correct"
        ],

        as_index=False
    )

    .agg(

        N=(
            "md5hash",
            "count"
        ),

        Entropy_Mean=(
            "Ensemble_Entropy",
            "mean"
        ),

        Entropy_Min=(
            "Ensemble_Entropy",
            "min"
        ),

        Entropy_Max=(
            "Ensemble_Entropy",
            "max"
        )
    )
)


case_availability_df[
    "Class"
] = (

    case_availability_df[
        "target"
    ]

    .map({

        0:
            "Benign",

        1:
            "Malignant"
    })
)


case_availability_df[
    "Outcome"
] = (

    case_availability_df[
        "Correct"
    ]

    .map({

        True:
            "Correct",

        False:
            "Incorrect"
    })
)


display(

    case_availability_df[
        [
            "Evaluation_Domain",
            "Class",
            "Outcome",
            "N",
            "Entropy_Mean",
            "Entropy_Min",
            "Entropy_Max"
        ]
    ]
)

In [ ]:
# ============================================
# Section 5.3
# Audit Fitzpatrick VI XAI Cases
# ============================================

fitz6_cases_df = (

    phase8_case_df[

        phase8_case_df[
            "fitzpatrick_scale"
        ]
        ==
        6
    ]

    .copy()

    .sort_values(
        "Ensemble_Entropy",
        ascending=False
    )
)


print(
    "Fitzpatrick VI images:",
    len(
        fitz6_cases_df
    )
)


print(
    "\nClass distribution:"
)


print(

    fitz6_cases_df[
        "target"
    ]

    .value_counts()
    .sort_index()
)


print(
    "\nCorrectness:"
)


print(

    fitz6_cases_df[
        "Correct"
    ]
    .value_counts()
)


display(

    fitz6_cases_df[
        [
            "md5hash",
            "target",
            "Ensemble_Probability",
            "Ensemble_Entropy",
            "Probability_SD",
            "Correct"
        ]
    ]

    .head(
        10
    )
)

In [ ]:
# ============================================
# Section 6.1
# Deterministic Core XAI Case Selection
# ============================================

selected_case_rows = []


for domain_name in [
    "Fair",
    "Dark"
]:

    for target_value in [
        0,
        1
    ]:

        class_name = (
            "Benign"
            if target_value == 0
            else "Malignant"
        )


        subgroup_df = (

            phase8_case_df[

                (
                    phase8_case_df[
                        "Evaluation_Domain"
                    ]
                    ==
                    domain_name
                )

                &

                (
                    phase8_case_df[
                        "target"
                    ]
                    ==
                    target_value
                )
            ]

            .copy()
        )


        # ------------------------------------
        # Correct cases
        # ------------------------------------

        correct_df = (

            subgroup_df[

                subgroup_df[
                    "Correct"
                ]
                ==
                True
            ]

            .copy()
        )


        low_entropy_case = (

            correct_df

            .sort_values(
                [
                    "Ensemble_Entropy",
                    "md5hash"
                ],
                ascending=[
                    True,
                    True
                ]
            )

            .iloc[0]
        )


        high_entropy_case = (

            correct_df

            .sort_values(
                [
                    "Ensemble_Entropy",
                    "md5hash"
                ],
                ascending=[
                    False,
                    True
                ]
            )

            .iloc[0]
        )


        for case_type, case_row in [

            (
                "Correct_LowEntropy",
                low_entropy_case
            ),

            (
                "Correct_HighEntropy",
                high_entropy_case
            )

        ]:

            selected_case_rows.append({

                "Selection_Group":
                    "Core",

                "Domain":
                    domain_name,

                "Class":
                    class_name,

                "Target":
                    int(
                        target_value
                    ),

                "Case_Type":
                    case_type,

                "md5hash":
                    case_row[
                        "md5hash"
                    ],

                "Image_Path":
                    case_row[
                        "image_path"
                    ],

                "Fitzpatrick":
                    int(
                        case_row[
                            "fitzpatrick_scale"
                        ]
                    ),

                "Ensemble_Probability":
                    float(
                        case_row[
                            "Ensemble_Probability"
                        ]
                    ),

                "Ensemble_Entropy":
                    float(
                        case_row[
                            "Ensemble_Entropy"
                        ]
                    ),

                "Probability_SD":
                    float(
                        case_row[
                            "Probability_SD"
                        ]
                    ),

                "Correct":
                    True
            })


        # ------------------------------------
        # Strong-confidence error
        # ------------------------------------

        incorrect_df = (

            subgroup_df[

                subgroup_df[
                    "Correct"
                ]
                ==
                False
            ]

            .copy()
        )


        if target_value == 0:

            # Benign falsely predicted malignant:
            # strongest error = highest malignant probability

            error_case = (

                incorrect_df

                .sort_values(
                    [
                        "Ensemble_Probability",
                        "md5hash"
                    ],
                    ascending=[
                        False,
                        True
                    ]
                )

                .iloc[0]
            )

            error_type = (
                "Incorrect_HighConfidence_FP"
            )

        else:

            # Malignant falsely predicted benign:
            # strongest error = lowest malignant probability

            error_case = (

                incorrect_df

                .sort_values(
                    [
                        "Ensemble_Probability",
                        "md5hash"
                    ],
                    ascending=[
                        True,
                        True
                    ]
                )

                .iloc[0]
            )

            error_type = (
                "Incorrect_HighConfidence_FN"
            )


        selected_case_rows.append({

            "Selection_Group":
                "Core",

            "Domain":
                domain_name,

            "Class":
                class_name,

            "Target":
                int(
                    target_value
                ),

            "Case_Type":
                error_type,

            "md5hash":
                error_case[
                    "md5hash"
                ],

            "Image_Path":
                error_case[
                    "image_path"
                ],

            "Fitzpatrick":
                int(
                    error_case[
                        "fitzpatrick_scale"
                    ]
                ),

            "Ensemble_Probability":
                float(
                    error_case[
                        "Ensemble_Probability"
                    ]
                ),

            "Ensemble_Entropy":
                float(
                    error_case[
                        "Ensemble_Entropy"
                    ]
                ),

            "Probability_SD":
                float(
                    error_case[
                        "Probability_SD"
                    ]
                ),

            "Correct":
                False
        })


phase8_core_cases_df = (
    pd.DataFrame(
        selected_case_rows
    )
)


display(
    phase8_core_cases_df
)


print(
    "\nCore cases selected:",
    len(
        phase8_core_cases_df
    )
)

In [ ]:
# ============================================
# Section 6.2
# Select Fitzpatrick VI Targeted XAI Cases
# ============================================

fitz6_ranked_df = (

    phase8_case_df[

        phase8_case_df[
            "fitzpatrick_scale"
        ]
        ==
        6
    ]

    .copy()

    .sort_values(
        [
            "Ensemble_Entropy",
            "md5hash"
        ],
        ascending=[
            False,
            True
        ]
    )
)


fitz6_selected_rows = []


# Highest entropy overall
top_overall = (
    fitz6_ranked_df
    .iloc[0]
)


fitz6_selected_rows.append(
    top_overall
)


# Highest entropy incorrect case
incorrect_fitz6 = (

    fitz6_ranked_df[

        fitz6_ranked_df[
            "Correct"
        ]
        ==
        False
    ]
)


if len(
    incorrect_fitz6
) > 0:

    top_incorrect = (
        incorrect_fitz6
        .iloc[0]
    )

    if (
        top_incorrect[
            "md5hash"
        ]
        !=
        top_overall[
            "md5hash"
        ]
    ):

        fitz6_selected_rows.append(
            top_incorrect
        )


# Highest entropy remaining case
for _, row in (
    fitz6_ranked_df.iterrows()
):

    existing_hashes = [

        selected_row[
            "md5hash"
        ]

        for selected_row
        in fitz6_selected_rows
    ]


    if (
        row[
            "md5hash"
        ]
        not in existing_hashes
    ):

        fitz6_selected_rows.append(
            row
        )


    if len(
        fitz6_selected_rows
    ) >= 3:

        break


phase8_fitz6_cases_df = pd.DataFrame(
    fitz6_selected_rows
)


phase8_fitz6_cases_df = (

    phase8_fitz6_cases_df[
        [
            "md5hash",
            "image_path",
            "target",
            "Evaluation_Domain",
            "fitzpatrick_scale",
            "Ensemble_Probability",
            "Ensemble_Entropy",
            "Probability_SD",
            "Correct"
        ]
    ]

    .rename(
        columns={

            "image_path":
                "Image_Path",

            "target":
                "Target",

            "Evaluation_Domain":
                "Domain",

            "fitzpatrick_scale":
                "Fitzpatrick"
        }
    )
)


phase8_fitz6_cases_df[
    "Selection_Group"
] = "Fitzpatrick_VI_Targeted"


phase8_fitz6_cases_df[
    "Class"
] = (

    phase8_fitz6_cases_df[
        "Target"
    ]

    .map({

        0:
            "Benign",

        1:
            "Malignant"
    })
)


phase8_fitz6_cases_df[
    "Case_Type"
] = (
    "HighEntropy_FitzpatrickVI"
)


display(
    phase8_fitz6_cases_df
)

In [ ]:
# ============================================
# Section 6.3
# Freeze Final XAI Case List
# ============================================

common_columns = [

    "Selection_Group",
    "Domain",
    "Class",
    "Target",
    "Case_Type",
    "md5hash",
    "Image_Path",
    "Fitzpatrick",
    "Ensemble_Probability",
    "Ensemble_Entropy",
    "Probability_SD",
    "Correct"
]


phase8_selected_cases_df = pd.concat(

    [

        phase8_core_cases_df[
            common_columns
        ],

        phase8_fitz6_cases_df[
            common_columns
        ]
    ],

    ignore_index=True
)


phase8_selected_cases_df = (

    phase8_selected_cases_df

    .drop_duplicates(
        subset=[
            "md5hash"
        ],
        keep="first"
    )

    .reset_index(
        drop=True
    )
)


phase8_selected_cases_df[
    "Case_ID"
] = [

    f"XAI_{index + 1:02d}"

    for index in range(
        len(
            phase8_selected_cases_df
        )
    )
]


phase8_selected_cases_df = (

    phase8_selected_cases_df[
        [
            "Case_ID",
            *common_columns
        ]
    ]
)


PHASE8_SELECTED_CASES_FILE = (

    PHASE8_TABLE_PATH
    / "Phase8_Frozen_XAI_Cases.csv"
)


phase8_selected_cases_df.to_csv(

    PHASE8_SELECTED_CASES_FILE,

    index=False
)


display(
    phase8_selected_cases_df
)


print(
    "\nFrozen XAI cases:",
    len(
        phase8_selected_cases_df
    )
)


print(
    "\n✓ Case list frozen before Grad-CAM++ generation."
)

print(
    PHASE8_SELECTED_CASES_FILE
)

In [ ]:
# ============================================
# Section 7.1
# Restore Phase 6 ResNet50 Architecture
# ============================================

class ResNet50WithFeatures(
    nn.Module
):

    def __init__(
        self,
        pretrained=False
    ):

        super().__init__()


        weights = (

            ResNet50_Weights.DEFAULT

            if pretrained

            else None
        )


        backbone = models.resnet50(
            weights=weights
        )


        self.feature_extractor = (
            nn.Sequential(

                backbone.conv1,
                backbone.bn1,
                backbone.relu,
                backbone.maxpool,

                backbone.layer1,
                backbone.layer2,
                backbone.layer3,
                backbone.layer4,

                backbone.avgpool
            )
        )


        self.classifier = nn.Linear(
            2048,
            2
        )


    def forward(
        self,
        x
    ):

        feature_map = x


        # conv1 -> layer4
        for layer in list(
            self.feature_extractor.children()
        )[:-1]:

            feature_map = layer(
                feature_map
            )


        pooled = (

            self.feature_extractor[-1](
                feature_map
            )

            .flatten(
                1
            )
        )


        logits = self.classifier(
            pooled
        )


        return (
            logits,
            pooled
        )


print(
    "✓ ResNet50WithFeatures restored."
)

In [ ]:
# ============================================
# Section 7.2
# Grad-CAM++ Compatible ResNet50
# ============================================

class ResNet50GradCAM(
    nn.Module
):

    def __init__(
        self
    ):

        super().__init__()


        backbone = models.resnet50(
            weights=None
        )


        self.conv1 = backbone.conv1
        self.bn1 = backbone.bn1
        self.relu = backbone.relu
        self.maxpool = backbone.maxpool

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        self.avgpool = backbone.avgpool

        self.classifier = nn.Linear(
            2048,
            2
        )


    def forward(
        self,
        x
    ):

        x = self.conv1(
            x
        )

        x = self.bn1(
            x
        )

        x = self.relu(
            x
        )

        x = self.maxpool(
            x
        )

        x = self.layer1(
            x
        )

        x = self.layer2(
            x
        )

        x = self.layer3(
            x
        )

        feature_map = self.layer4(
            x
        )


        pooled = self.avgpool(
            feature_map
        )


        pooled = torch.flatten(
            pooled,
            1
        )


        logits = self.classifier(
            pooled
        )


        return (
            logits,
            feature_map
        )


print(
    "✓ Grad-CAM++ compatible model defined."
)

In [ ]:
# ============================================
# Section 7.3
# Load Frozen Phase 6 Checkpoint for XAI
# ============================================

PHASE6_TUNING_PATH = (
    PHASE6_PATH
    / "tuning"
)


def lambda_to_name(
    lambda_value
):

    return str(
        lambda_value
    ).replace(
        ".",
        "p"
    )


def phase8_checkpoint_path(
    seed
):

    return (

        PHASE6_TUNING_PATH

        / (
            f"lambda_"
            f"{lambda_to_name(0.0)}"
            f"_seed_{seed}"
        )

        / "best_model.pth"
    )


def load_phase8_model(
    seed
):

    checkpoint_path = (
        phase8_checkpoint_path(
            seed
        )
    )


    assert (
        checkpoint_path.exists()
    ), (
        f"Missing checkpoint: "
        f"{checkpoint_path}"
    )


    checkpoint = torch.load(

        checkpoint_path,

        map_location=DEVICE,

        weights_only=False
    )


    original_state = checkpoint[
        "model_state_dict"
    ]


    xai_state = {}


    prefix_mapping = {

        "feature_extractor.0.":
            "conv1.",

        "feature_extractor.1.":
            "bn1.",

        "feature_extractor.4.":
            "layer1.",

        "feature_extractor.5.":
            "layer2.",

        "feature_extractor.6.":
            "layer3.",

        "feature_extractor.7.":
            "layer4.",

        "classifier.":
            "classifier."
    }


    for original_key, value in (
        original_state.items()
    ):

        mapped = False


        for old_prefix, new_prefix in (
            prefix_mapping.items()
        ):

            if original_key.startswith(
                old_prefix
            ):

                new_key = (

                    new_prefix

                    +

                    original_key[
                        len(
                            old_prefix
                        ):
                    ]
                )


                xai_state[
                    new_key
                ] = value

                mapped = True

                break


    model = ResNet50GradCAM()


    load_result = model.load_state_dict(

        xai_state,

        strict=True
    )


    model = model.to(
        DEVICE
    )


    model.eval()


    return model


# Sanity check one seed
test_xai_model = load_phase8_model(
    42
)


print(
    "✓ Seed 42 XAI model loaded successfully."
)


del test_xai_model

gc.collect()

if torch.cuda.is_available():

    torch.cuda.empty_cache()

In [ ]:
# ============================================
# Section 8.1
# Grad-CAM++ Implementation
# ============================================

def generate_gradcam_plus_plus(
    model,
    input_tensor,
    target_class
):

    model.zero_grad(
        set_to_none=True
    )


    input_tensor = (

        input_tensor
        .to(
            DEVICE
        )
    )


    logits, feature_map = model(
        input_tensor
    )


    feature_map.retain_grad()


    target_score = (

        logits[
            0,
            target_class
        ]
    )


    target_score.backward(
        retain_graph=False
    )


    gradients = (
        feature_map.grad
    )


    activations = (
        feature_map
    )


    gradients_2 = (
        gradients ** 2
    )


    gradients_3 = (
        gradients ** 3
    )


    sum_activations = (

        activations.sum(
            dim=(
                2,
                3
            ),

            keepdim=True
        )
    )


    denominator = (

        2.0
        *
        gradients_2

        +

        sum_activations
        *
        gradients_3
    )


    denominator = torch.where(

        denominator
        !=
        0.0,

        denominator,

        torch.ones_like(
            denominator
        )
    )


    alpha = (

        gradients_2
        /
        denominator
    )


    positive_gradients = F.relu(
        gradients
    )


    weights = (

        alpha
        *
        positive_gradients

    ).sum(

        dim=(
            2,
            3
        ),

        keepdim=True
    )


    cam = (

        weights
        *
        activations

    ).sum(

        dim=1,
        keepdim=True
    )


    cam = F.relu(
        cam
    )


    cam = F.interpolate(

        cam,

        size=(
            IMAGE_SIZE,
            IMAGE_SIZE
        ),

        mode="bilinear",

        align_corners=False
    )


    cam = (
        cam[
            0,
            0
        ]
        .detach()
        .cpu()
        .numpy()
    )


    cam_min = float(
        cam.min()
    )


    cam_max = float(
        cam.max()
    )


    if (
        cam_max
        >
        cam_min
    ):

        cam = (

            cam
            -
            cam_min

        ) / (

            cam_max
            -
            cam_min
        )

    else:

        cam = np.zeros_like(
            cam
        )


    probability = (

        torch.softmax(
            logits,
            dim=1
        )[
            0,
            1
        ]

        .detach()
        .cpu()
        .item()
    )


    return (
        cam,
        probability
    )

In [ ]:
# ============================================
# Section 8.2
# Load One Image for Grad-CAM++
# ============================================

def load_phase8_image(
    image_path
):

    original_image = Image.open(
        image_path
    ).convert(
        "RGB"
    )


    resized_image = original_image.resize(

        (
            IMAGE_SIZE,
            IMAGE_SIZE
        )
    )


    input_tensor = eval_transform(
        original_image
    ).unsqueeze(
        0
    )


    return (
        original_image,
        resized_image,
        input_tensor
    )


print(
    "✓ Phase 8 image loader ready."
)

In [ ]:
# ============================================
# Section 9.1
# Generate Five-Seed Ensemble Grad-CAM++ Maps
# ============================================

def normalize_cam(
    cam
):

    cam = np.asarray(
        cam,
        dtype=np.float32
    )


    cam_min = float(
        np.min(
            cam
        )
    )


    cam_max = float(
        np.max(
            cam
        )
    )


    if (
        cam_max
        >
        cam_min
    ):

        cam = (

            cam
            -
            cam_min

        ) / (

            cam_max
            -
            cam_min
        )

    else:

        cam = np.zeros_like(
            cam
        )


    return cam


def generate_ensemble_gradcam_for_case(
    case_row
):

    image_path = (
        case_row[
            "Image_Path"
        ]
    )


    ensemble_probability = float(
        case_row[
            "Ensemble_Probability"
        ]
    )


    predicted_class = (

        1

        if ensemble_probability
        >=
        0.5

        else 0
    )


    (
        original_image,
        resized_image,
        input_tensor

    ) = load_phase8_image(
        image_path
    )


    seed_cams = []

    seed_probabilities = []


    for seed in PHASE8_SEEDS:

        model = load_phase8_model(
            seed
        )


        cam, probability = (
            generate_gradcam_plus_plus(

                model=model,

                input_tensor=input_tensor,

                target_class=predicted_class
            )
        )


        cam = normalize_cam(
            cam
        )


        seed_cams.append(
            cam
        )


        seed_probabilities.append(
            probability
        )


        del model

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


    seed_cams_array = np.stack(
        seed_cams,
        axis=0
    )


    ensemble_cam = np.mean(
        seed_cams_array,
        axis=0
    )


    ensemble_cam = normalize_cam(
        ensemble_cam
    )


    cam_sd = np.std(
        seed_cams_array,
        axis=0,
        ddof=1
    )


    return {

        "Original_Image":
            original_image,

        "Resized_Image":
            resized_image,

        "Predicted_Class":
            predicted_class,

        "Seed_Probabilities":
            seed_probabilities,

        "Seed_CAMs":
            seed_cams_array,

        "Ensemble_CAM":
            ensemble_cam,

        "CAM_SD":
            cam_sd
    }


print(
    "✓ Five-seed ensemble Grad-CAM++ function ready."
)

In [ ]:
# ============================================
# Section 9.2
# Generate Grad-CAM++ for All Frozen Cases
# ============================================

phase8_gradcam_results = {}


for _, case_row in (
    phase8_selected_cases_df.iterrows()
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    print(
        f"Generating Grad-CAM++: "
        f"{case_id}"
    )


    result = (
        generate_ensemble_gradcam_for_case(
            case_row
        )
    )


    phase8_gradcam_results[
        case_id
    ] = result


print(
    "\nGenerated cases:",
    len(
        phase8_gradcam_results
    )
)


assert (
    len(
        phase8_gradcam_results
    )
    ==
    len(
        phase8_selected_cases_df
    )
)


print(
    "✓ All frozen Grad-CAM++ cases generated."
)

In [ ]:
# ============================================
# Section 9.3
# Save Grad-CAM++ Arrays
# ============================================

saved_cam_rows = []


for _, case_row in (
    phase8_selected_cases_df.iterrows()
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    result = (
        phase8_gradcam_results[
            case_id
        ]
    )


    ensemble_cam_file = (

        PHASE8_HEATMAP_PATH
        / f"{case_id}_ensemble_cam.npy"
    )


    cam_sd_file = (

        PHASE8_HEATMAP_PATH
        / f"{case_id}_cam_sd.npy"
    )


    np.save(
        ensemble_cam_file,
        result[
            "Ensemble_CAM"
        ]
    )


    np.save(
        cam_sd_file,
        result[
            "CAM_SD"
        ]
    )


    saved_cam_rows.append({

        "Case_ID":
            case_id,

        "Ensemble_CAM_File":
            str(
                ensemble_cam_file
            ),

        "CAM_SD_File":
            str(
                cam_sd_file
            )
    })


phase8_saved_cam_df = (
    pd.DataFrame(
        saved_cam_rows
    )
)


display(
    phase8_saved_cam_df
)


print(
    "✓ Numerical Grad-CAM++ arrays saved."
)

In [ ]:
# ============================================
# Section 10.1
# Grad-CAM++ Overlay Helper
# ============================================

def create_cam_overlay(
    resized_image,
    cam,
    alpha=0.45
):

    image_array = (

        np.asarray(
            resized_image
        )

        .astype(
            np.float32
        )

        /
        255.0
    )


    cmap = plt.get_cmap(
        "jet"
    )


    heatmap_rgb = (

        cmap(
            cam
        )[
            ...,
            :3
        ]
    )


    overlay = (

        (
            1.0
            -
            alpha
        )
        *
        image_array

        +

        alpha
        *
        heatmap_rgb
    )


    overlay = np.clip(
        overlay,
        0.0,
        1.0
    )


    return overlay


print(
    "✓ Overlay helper ready."
)

In [ ]:
# ============================================
# Section 10.2
# Save Individual Grad-CAM++ Case Figures
# ============================================

figure_rows = []


for _, case_row in (
    phase8_selected_cases_df.iterrows()
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    result = (
        phase8_gradcam_results[
            case_id
        ]
    )


    resized_image = (
        result[
            "Resized_Image"
        ]
    )


    ensemble_cam = (
        result[
            "Ensemble_CAM"
        ]
    )


    cam_sd = (
        result[
            "CAM_SD"
        ]
    )


    overlay = create_cam_overlay(

        resized_image=resized_image,

        cam=ensemble_cam,

        alpha=0.45
    )


    predicted_class = int(
        result[
            "Predicted_Class"
        ]
    )


    predicted_name = (

        "Malignant"

        if predicted_class == 1

        else "Benign"
    )


    true_name = (
        case_row[
            "Class"
        ]
    )


    fig = plt.figure(
        figsize=(
            14,
            4
        )
    )


    ax1 = fig.add_subplot(
        1,
        4,
        1
    )

    ax1.imshow(
        resized_image
    )

    ax1.set_title(
        "Original"
    )

    ax1.axis(
        "off"
    )


    ax2 = fig.add_subplot(
        1,
        4,
        2
    )

    image2 = ax2.imshow(
        ensemble_cam,
        vmin=0.0,
        vmax=1.0
    )

    ax2.set_title(
        "Ensemble Grad-CAM++"
    )

    ax2.axis(
        "off"
    )


    ax3 = fig.add_subplot(
        1,
        4,
        3
    )

    ax3.imshow(
        overlay
    )

    ax3.set_title(
        "Overlay"
    )

    ax3.axis(
        "off"
    )


    ax4 = fig.add_subplot(
        1,
        4,
        4
    )

    image4 = ax4.imshow(
        cam_sd
    )

    ax4.set_title(
        "CAM variability"
    )

    ax4.axis(
        "off"
    )


    fig.suptitle(

        (
            f"{case_id} | "
            f"{case_row['Domain']} | "
            f"Fitzpatrick {case_row['Fitzpatrick']} | "
            f"True: {true_name} | "
            f"Predicted: {predicted_name} | "
            f"p={case_row['Ensemble_Probability']:.3f} | "
            f"H={case_row['Ensemble_Entropy']:.3f}"
        ),

        fontsize=11
    )


    fig.tight_layout()


    figure_file = (

        PHASE8_FIGURE_PATH
        / f"{case_id}_GradCAMPP.png"
    )


    fig.savefig(

        figure_file,

        dpi=300,

        bbox_inches="tight"
    )


    plt.close(
        fig
    )


    figure_rows.append({

        "Case_ID":
            case_id,

        "Figure_File":
            str(
                figure_file
            )
    })


phase8_case_figure_df = (
    pd.DataFrame(
        figure_rows
    )
)


print(
    "Figures saved:",
    len(
        phase8_case_figure_df
    )
)


print(
    "✓ Individual Grad-CAM++ figures saved."
)

In [ ]:
# ============================================
# Section 11.1
# Quantify Seed-to-Seed CAM Stability
# ============================================

from itertools import combinations


stability_rows = []


for _, case_row in (
    phase8_selected_cases_df.iterrows()
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    seed_cams = (

        phase8_gradcam_results[
            case_id
        ][
            "Seed_CAMs"
        ]
    )


    correlations = []


    for i, j in combinations(
        range(
            len(
                PHASE8_SEEDS
            )
        ),
        2
    ):

        cam_i = (
            seed_cams[
                i
            ]
            .reshape(
                -1
            )
        )


        cam_j = (
            seed_cams[
                j
            ]
            .reshape(
                -1
            )
        )


        if (
            np.std(
                cam_i
            )
            ==
            0

            or

            np.std(
                cam_j
            )
            ==
            0
        ):

            correlation = np.nan

        else:

            correlation = float(

                np.corrcoef(
                    cam_i,
                    cam_j
                )[
                    0,
                    1
                ]
            )


        correlations.append(
            correlation
        )


    correlations = np.asarray(
        correlations,
        dtype=float
    )


    mean_cam_sd = float(

        np.mean(

            phase8_gradcam_results[
                case_id
            ][
                "CAM_SD"
            ]
        )
    )


    stability_rows.append({

        "Case_ID":
            case_id,

        "Domain":
            case_row[
                "Domain"
            ],

        "Class":
            case_row[
                "Class"
            ],

        "Case_Type":
            case_row[
                "Case_Type"
            ],

        "Fitzpatrick":
            int(
                case_row[
                    "Fitzpatrick"
                ]
            ),

        "Ensemble_Entropy":
            float(
                case_row[
                    "Ensemble_Entropy"
                ]
            ),

        "Probability_SD":
            float(
                case_row[
                    "Probability_SD"
                ]
            ),

        "Mean_CAM_Correlation":
            float(
                np.nanmean(
                    correlations
                )
            ),

        "Min_CAM_Correlation":
            float(
                np.nanmin(
                    correlations
                )
            ),

        "Mean_CAM_SD":
            mean_cam_sd
    })


phase8_cam_stability_df = (
    pd.DataFrame(
        stability_rows
    )
)


display(
    phase8_cam_stability_df
)

In [ ]:
# ============================================
# Section 11.2
# Exploratory Uncertainty-XAI Stability Association
# ============================================

from scipy.stats import (
    spearmanr
)


entropy_vs_camcorr = spearmanr(

    phase8_cam_stability_df[
        "Ensemble_Entropy"
    ],

    phase8_cam_stability_df[
        "Mean_CAM_Correlation"
    ]
)


probabilitysd_vs_camcorr = spearmanr(

    phase8_cam_stability_df[
        "Probability_SD"
    ],

    phase8_cam_stability_df[
        "Mean_CAM_Correlation"
    ]
)


entropy_vs_camsd = spearmanr(

    phase8_cam_stability_df[
        "Ensemble_Entropy"
    ],

    phase8_cam_stability_df[
        "Mean_CAM_SD"
    ]
)


phase8_xai_association_df = pd.DataFrame([

    {

        "Association":
            (
                "Ensemble entropy vs "
                "mean CAM correlation"
            ),

        "Spearman_Rho":
            float(
                entropy_vs_camcorr.statistic
            ),

        "P_Value":
            float(
                entropy_vs_camcorr.pvalue
            )
    },

    {

        "Association":
            (
                "Probability SD vs "
                "mean CAM correlation"
            ),

        "Spearman_Rho":
            float(
                probabilitysd_vs_camcorr.statistic
            ),

        "P_Value":
            float(
                probabilitysd_vs_camcorr.pvalue
            )
    },

    {

        "Association":
            (
                "Ensemble entropy vs "
                "mean CAM spatial SD"
            ),

        "Spearman_Rho":
            float(
                entropy_vs_camsd.statistic
            ),

        "P_Value":
            float(
                entropy_vs_camsd.pvalue
            )
    }
])


display(
    phase8_xai_association_df
)

In [ ]:
# ============================================
# Section 11.3
# Save Quantitative XAI Results
# ============================================

phase8_cam_stability_df.to_csv(

    PHASE8_TABLE_PATH
    / "Phase8_CAM_Stability.csv",

    index=False
)


phase8_xai_association_df.to_csv(

    PHASE8_TABLE_PATH
    / "Phase8_Uncertainty_CAM_Associations.csv",

    index=False
)


phase8_case_figure_df.to_csv(

    PHASE8_TABLE_PATH
    / "Phase8_Case_Figure_Index.csv",

    index=False
)


print(
    "✓ Quantitative XAI outputs saved."
)

In [ ]:
# ============================================
# Section 12.1
# Freeze Grad-CAM++ Visual / Spatial Audit Plan
# ============================================

phase8_visual_audit_plan = {

    "Quantitative_Spatial_Metrics": [

        "CAM center of mass",

        "central-region attention fraction",

        "border attention fraction",

        "high-activation area fraction",

        "CAM concentration",

        "seed-to-seed CAM correlation"
    ],

    "Central_Region":
        "Middle 50% of image width and height",

    "Border_Region":
        "Outer 15% of image dimensions",

    "High_Activation_Threshold":
        0.5,

    "Important_Limitation":
        (
            "No lesion segmentation masks are available. "
            "Therefore spatial metrics measure attention "
            "distribution, not verified lesion overlap."
        ),

    "Visual_Audit_Categories": [

        "Predominantly lesion-focused",

        "Mixed lesion and surrounding skin",

        "Predominantly background/artifact-focused",

        "Diffuse / non-localized",

        "Unable to determine"
    ],

    "Visual_Inspection_Not_Used_For_Model_Selection":
        True,

    "Case_List_Remains_Frozen":
        True,

    "Frozen":
        True
}


PHASE8_VISUAL_AUDIT_PLAN_FILE = (

    PHASE8_CONFIG_PATH
    / "Phase8_Visual_Audit_Plan.json"
)


with open(
    PHASE8_VISUAL_AUDIT_PLAN_FILE,
    "w"
) as f:

    json.dump(
        phase8_visual_audit_plan,
        f,
        indent=4
    )


print(
    "✓ Phase 8 spatial/visual audit plan frozen."
)

In [ ]:
# ============================================
# Section 12.2
# Quantify Grad-CAM++ Spatial Characteristics
# ============================================

def compute_cam_spatial_metrics(
    cam,
    high_threshold=0.5
):

    cam = np.asarray(
        cam,
        dtype=np.float64
    )


    height, width = cam.shape


    total_attention = (
        cam.sum()
    )


    if total_attention <= 0:

        return {

            "CAM_Center_X":
                np.nan,

            "CAM_Center_Y":
                np.nan,

            "Central_Attention_Fraction":
                np.nan,

            "Border_Attention_Fraction":
                np.nan,

            "High_Activation_Area_Fraction":
                0.0,

            "CAM_Concentration":
                np.nan
        }


    # ----------------------------------------
    # Normalized center of mass
    # ----------------------------------------

    y_coordinates, x_coordinates = (
        np.mgrid[
            0:height,
            0:width
        ]
    )


    center_x = (

        (
            cam
            *
            x_coordinates
        ).sum()

        /
        total_attention

        /
        (
            width - 1
        )
    )


    center_y = (

        (
            cam
            *
            y_coordinates
        ).sum()

        /
        total_attention

        /
        (
            height - 1
        )
    )


    # ----------------------------------------
    # Central 50% region
    # ----------------------------------------

    x_start = int(
        width * 0.25
    )

    x_end = int(
        width * 0.75
    )

    y_start = int(
        height * 0.25
    )

    y_end = int(
        height * 0.75
    )


    central_attention = (

        cam[
            y_start:y_end,
            x_start:x_end
        ]
        .sum()

        /
        total_attention
    )


    # ----------------------------------------
    # Outer 15% border
    # ----------------------------------------

    border_y = int(
        height * 0.15
    )

    border_x = int(
        width * 0.15
    )


    border_mask = np.zeros_like(
        cam,
        dtype=bool
    )


    border_mask[
        :border_y,
        :
    ] = True

    border_mask[
        -border_y:,
        :
    ] = True

    border_mask[
        :,
        :border_x
    ] = True

    border_mask[
        :,
        -border_x:
    ] = True


    border_attention = (

        cam[
            border_mask
        ]
        .sum()

        /
        total_attention
    )


    # ----------------------------------------
    # High-activation area
    # ----------------------------------------

    high_activation_fraction = float(

        (
            cam
            >=
            high_threshold
        )
        .mean()
    )


    # ----------------------------------------
    # Concentration index
    #
    # Mean squared normalized attention.
    # Higher = more spatially concentrated.
    # ----------------------------------------

    normalized_attention = (

        cam
        /
        total_attention
    )


    concentration = float(

        np.sum(
            normalized_attention ** 2
        )

        *
        cam.size
    )


    return {

        "CAM_Center_X":
            float(
                center_x
            ),

        "CAM_Center_Y":
            float(
                center_y
            ),

        "Central_Attention_Fraction":
            float(
                central_attention
            ),

        "Border_Attention_Fraction":
            float(
                border_attention
            ),

        "High_Activation_Area_Fraction":
            high_activation_fraction,

        "CAM_Concentration":
            concentration
    }

In [ ]:
# ============================================
# Section 12.3
# Build Grad-CAM++ Spatial Audit Table
# ============================================

spatial_rows = []


for _, case_row in (
    phase8_selected_cases_df.iterrows()
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    cam = (

        phase8_gradcam_results[
            case_id
        ][
            "Ensemble_CAM"
        ]
    )


    spatial_metrics = (
        compute_cam_spatial_metrics(
            cam
        )
    )


    stability_row = (

        phase8_cam_stability_df[

            phase8_cam_stability_df[
                "Case_ID"
            ]
            ==
            case_id

        ]

        .iloc[0]
    )


    spatial_rows.append({

        "Case_ID":
            case_id,

        "Domain":
            case_row[
                "Domain"
            ],

        "Class":
            case_row[
                "Class"
            ],

        "Case_Type":
            case_row[
                "Case_Type"
            ],

        "Fitzpatrick":
            int(
                case_row[
                    "Fitzpatrick"
                ]
            ),

        "Correct":
            bool(
                case_row[
                    "Correct"
                ]
            ),

        "Ensemble_Entropy":
            float(
                case_row[
                    "Ensemble_Entropy"
                ]
            ),

        "Probability_SD":
            float(
                case_row[
                    "Probability_SD"
                ]
            ),

        "Mean_CAM_Correlation":
            float(
                stability_row[
                    "Mean_CAM_Correlation"
                ]
            ),

        **spatial_metrics
    })


phase8_spatial_audit_df = (
    pd.DataFrame(
        spatial_rows
    )
)


display(
    phase8_spatial_audit_df
)

In [ ]:
# ============================================
# Section 12.4
# Correct vs Incorrect XAI Stability Summary
# ============================================

xai_outcome_summary = (

    phase8_spatial_audit_df

    .groupby(
        "Correct"
    )

    .agg(

        N=(
            "Case_ID",
            "count"
        ),

        Mean_CAM_Correlation=(
            "Mean_CAM_Correlation",
            "mean"
        ),

        Mean_CAM_Correlation_SD=(
            "Mean_CAM_Correlation",
            "std"
        ),

        Mean_CAM_SD=(
            "High_Activation_Area_Fraction",
            "mean"
        ),

        Central_Attention_Mean=(
            "Central_Attention_Fraction",
            "mean"
        ),

        Border_Attention_Mean=(
            "Border_Attention_Fraction",
            "mean"
        ),

        Concentration_Mean=(
            "CAM_Concentration",
            "mean"
        )
    )

    .reset_index()
)


display(
    xai_outcome_summary
)

In [ ]:
# ============================================
# Section 12.4B
# Clean Correct-vs-Incorrect XAI Summary
# ============================================

xai_outcome_summary = (

    phase8_spatial_audit_df

    .groupby(
        "Correct"
    )

    .agg(

        N=(
            "Case_ID",
            "count"
        ),

        CAM_Correlation_Mean=(
            "Mean_CAM_Correlation",
            "mean"
        ),

        CAM_Correlation_SD=(
            "Mean_CAM_Correlation",
            "std"
        ),

        Central_Attention_Mean=(
            "Central_Attention_Fraction",
            "mean"
        ),

        Border_Attention_Mean=(
            "Border_Attention_Fraction",
            "mean"
        ),

        High_Activation_Area_Mean=(
            "High_Activation_Area_Fraction",
            "mean"
        ),

        CAM_Concentration_Mean=(
            "CAM_Concentration",
            "mean"
        )
    )

    .reset_index()
)


display(
    xai_outcome_summary
)

In [ ]:
# ============================================
# Section 13.1
# Create Frozen Grad-CAM++ Contact Sheet
# ============================================

number_cases = len(
    phase8_selected_cases_df
)


number_columns = 4

number_rows = int(
    np.ceil(
        number_cases
        /
        number_columns
    )
)


fig = plt.figure(

    figsize=(
        16,
        number_rows * 4
    )
)


for plot_index, (
    _,
    case_row

) in enumerate(

    phase8_selected_cases_df.iterrows(),

    start=1
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    result = (
        phase8_gradcam_results[
            case_id
        ]
    )


    overlay = create_cam_overlay(

        resized_image=result[
            "Resized_Image"
        ],

        cam=result[
            "Ensemble_CAM"
        ],

        alpha=0.45
    )


    axis = fig.add_subplot(

        number_rows,
        number_columns,
        plot_index
    )


    axis.imshow(
        overlay
    )


    prediction_name = (

        "M"

        if result[
            "Predicted_Class"
        ]
        ==
        1

        else "B"
    )


    true_name = (

        "M"

        if case_row[
            "Target"
        ]
        ==
        1

        else "B"
    )


    axis.set_title(

        (
            f"{case_id} | "
            f"{case_row['Domain']} | "
            f"F{case_row['Fitzpatrick']}\n"
            f"True={true_name}, "
            f"Pred={prediction_name}, "
            f"H={case_row['Ensemble_Entropy']:.3f}"
        ),

        fontsize=9
    )


    axis.axis(
        "off"
    )


fig.tight_layout()


PHASE8_CONTACT_SHEET_FILE = (

    PHASE8_FIGURE_PATH
    / "Phase8_Frozen_GradCAMPP_Contact_Sheet.png"
)


fig.savefig(

    PHASE8_CONTACT_SHEET_FILE,

    dpi=300,

    bbox_inches="tight"
)


plt.show()


print(
    "✓ Contact sheet saved:"
)

print(
    PHASE8_CONTACT_SHEET_FILE
)

In [ ]:
# ============================================
# Section 13.1B
# Create Viewer-Friendly Grad-CAM++ Contact Sheet
# Display Modification Only
# ============================================

from PIL import ImageFilter


number_cases = len(
    phase8_selected_cases_df
)


number_columns = 4


number_rows = int(
    np.ceil(
        number_cases
        /
        number_columns
    )
)


fig = plt.figure(

    figsize=(
        16,
        number_rows * 4
    )
)


for plot_index, (
    _,
    case_row

) in enumerate(

    phase8_selected_cases_df.iterrows(),

    start=1
):

    case_id = (
        case_row[
            "Case_ID"
        ]
    )


    result = (
        phase8_gradcam_results[
            case_id
        ]
    )


    # ----------------------------------------
    # Mildly soften clinical photograph
    # for viewer-friendly presentation only
    # ----------------------------------------

    softened_image = (

        result[
            "Resized_Image"
        ]

        .filter(
            ImageFilter.GaussianBlur(
                radius=2.0
            )
        )
    )


    overlay = create_cam_overlay(

        resized_image=softened_image,

        cam=result[
            "Ensemble_CAM"
        ],

        alpha=0.55
    )


    axis = fig.add_subplot(

        number_rows,
        number_columns,
        plot_index
    )


    axis.imshow(
        overlay
    )


    prediction_name = (

        "M"

        if result[
            "Predicted_Class"
        ]
        ==
        1

        else "B"
    )


    true_name = (

        "M"

        if case_row[
            "Target"
        ]
        ==
        1

        else "B"
    )


    axis.set_title(

        (
            f"{case_id} | "
            f"{case_row['Domain']} | "
            f"F{case_row['Fitzpatrick']}\n"
            f"True={true_name}, "
            f"Pred={prediction_name}, "
            f"H={case_row['Ensemble_Entropy']:.3f}"
        ),

        fontsize=9
    )


    axis.axis(
        "off"
    )


fig.suptitle(

    (
        "Grad-CAM++ Explainability Cases "
        "(viewer-friendly visualization)"
    ),

    fontsize=14
)


fig.tight_layout()


PHASE8_CONTACT_SHEET_FILE = (

    PHASE8_FIGURE_PATH
    / "Phase8_GradCAMPP_Contact_Sheet_Viewer_Friendly.png"
)


fig.savefig(

    PHASE8_CONTACT_SHEET_FILE,

    dpi=300,

    bbox_inches="tight"
)


plt.show()


print(
    "✓ Viewer-friendly contact sheet saved:"
)

print(
    PHASE8_CONTACT_SHEET_FILE
)

print(
    "\nIMPORTANT:"
)

print(
    "The 14 frozen analytical cases were NOT changed."
)

print(
    "Only the visualization was softened for presentation."
)

In [ ]:
# ============================================
# Section 13.2
# Prepare Standardized Visual Audit Sheet
# ============================================

phase8_visual_review_df = (

    phase8_selected_cases_df[
        [
            "Case_ID",
            "Domain",
            "Class",
            "Case_Type",
            "Fitzpatrick",
            "Correct",
            "Ensemble_Entropy"
        ]
    ]

    .copy()
)


phase8_visual_review_df[
    "Attention_Category"
] = ""


phase8_visual_review_df[
    "Primary_Attention_Region"
] = ""


phase8_visual_review_df[
    "Visible_Artifact_Attention"
] = ""


phase8_visual_review_df[
    "Reviewer_Confidence"
] = ""


phase8_visual_review_df[
    "Notes"
] = ""


PHASE8_VISUAL_REVIEW_FILE = (

    PHASE8_TABLE_PATH
    / "Phase8_Standardized_Visual_Audit.csv"
)


phase8_visual_review_df.to_csv(

    PHASE8_VISUAL_REVIEW_FILE,

    index=False
)


display(
    phase8_visual_review_df
)


print(
    "✓ Standardized visual-audit sheet created."
)

In [ ]:
# ============================================
# Section 14.0
# Google Drive Storage Audit
# NO FILES ARE DELETED
# ============================================

from pathlib import Path


ARTIFACT_PATH = Path(
    "/content/drive/MyDrive/Phase_3_Artifacts"
)


def directory_size_gb(
    directory
):

    total_bytes = 0


    for path in directory.rglob("*"):

        if path.is_file():

            try:

                total_bytes += (
                    path.stat().st_size
                )

            except:

                pass


    return (
        total_bytes
        /
        (1024 ** 3)
    )


phase_sizes = []


for directory in sorted(
    ARTIFACT_PATH.iterdir()
):

    if directory.is_dir():

        size_gb = (
            directory_size_gb(
                directory
            )
        )


        phase_sizes.append({

            "Folder":
                directory.name,

            "Size_GB":
                size_gb
        })


phase_size_df = (

    pd.DataFrame(
        phase_sizes
    )

    .sort_values(
        "Size_GB",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


display(
    phase_size_df
)

In [ ]:
# ============================================
# Section 14.0B
# Find Largest Artifact Files
# NO FILES ARE DELETED
# ============================================

largest_files = []


for path in ARTIFACT_PATH.rglob("*"):

    if path.is_file():

        try:

            size_mb = (

                path.stat().st_size
                /
                (1024 ** 2)
            )


            largest_files.append({

                "File":
                    str(
                        path
                    ),

                "Size_MB":
                    size_mb
            })

        except:

            pass


largest_files_df = (

    pd.DataFrame(
        largest_files
    )

    .sort_values(
        "Size_MB",
        ascending=False
    )

    .reset_index(
        drop=True
    )
)


display(
    largest_files_df.head(
        40
    )
)

In [ ]:
# ============================================
# Storage Audit
# Identify Large Phase 6 Checkpoints
# ============================================

PHASE6_STORAGE_PATH = (
    ARTIFACT_PATH
    / "Phase_6_Domain_Adaptation"
)


phase6_large_files = []


for path in PHASE6_STORAGE_PATH.rglob("*"):

    if path.is_file():

        size_mb = (
            path.stat().st_size
            /
            (1024 ** 2)
        )


        if size_mb >= 80:

            phase6_large_files.append({

                "Name":
                    path.name,

                "Parent":
                    path.parent.name,

                "Size_MB":
                    size_mb,

                "Full_Path":
                    str(
                        path
                    )
            })


phase6_large_files_df = (

    pd.DataFrame(
        phase6_large_files
    )

    .sort_values(
        [
            "Size_MB",
            "Name"
        ],
        ascending=[
            False,
            True
        ]
    )

    .reset_index(
        drop=True
    )
)


display(
    phase6_large_files_df
)

In [ ]:
# ============================================
# Storage Cleanup 1
# Audit Phase 6 Resumable Checkpoints
# BEFORE DELETION
# ============================================

PHASE6_PATH = (
    ARTIFACT_PATH
    / "Phase_6_Domain_Adaptation"
)


last_checkpoint_files = sorted(

    PHASE6_PATH.rglob(
        "last_checkpoint.pth"
    )
)


cleanup_rows = []


for path in last_checkpoint_files:

    size_mb = (

        path.stat().st_size
        /
        (1024 ** 2)
    )


    cleanup_rows.append({

        "File":
            str(
                path
            ),

        "Run":
            path.parent.name,

        "Size_MB":
            size_mb
    })


phase6_cleanup_df = pd.DataFrame(
    cleanup_rows
)


display(
    phase6_cleanup_df
)


total_mb = (
    phase6_cleanup_df[
        "Size_MB"
    ]
    .sum()
)


print(
    "Files eligible for deletion:",
    len(
        phase6_cleanup_df
    )
)


print(
    f"Total recoverable space: "
    f"{total_mb / 1024:.2f} GB"
)

In [ ]:
# ============================================
# Storage Cleanup 2
# Save Cleanup Manifest
# ============================================

PHASE6_CLEANUP_MANIFEST = (

    PHASE6_PATH
    / "manifests"
    / "Phase6_Last_Checkpoint_Cleanup.csv"
)


phase6_cleanup_df.to_csv(

    PHASE6_CLEANUP_MANIFEST,

    index=False
)


cleanup_manifest = {

    "Reason":
        (
            "Phase 6 is complete and frozen. "
            "Resumable last checkpoints are no longer "
            "required for training continuation."
        ),

    "Deleted_File_Type":
        "last_checkpoint.pth",

    "Best_Model_Checkpoints_Preserved":
        True,

    "Results_Tables_Preserved":
        True,

    "Prediction_Files_Preserved":
        True,

    "Run_Manifests_Preserved":
        True,

    "Number_Eligible_Files":
        int(
            len(
                phase6_cleanup_df
            )
        ),

    "Recoverable_GB":
        float(
            total_mb
            /
            1024
        )
}


with open(

    PHASE6_PATH
    / "manifests"
    / "Phase6_Storage_Cleanup_Manifest.json",

    "w"

) as f:

    json.dump(
        cleanup_manifest,
        f,
        indent=4
    )


print(
    "✓ Cleanup audit saved."
)

In [ ]:
# ============================================
# Storage Cleanup 3
# Delete ONLY Resumable Last Checkpoints
# ============================================

deleted_files = 0

deleted_bytes = 0


for path in last_checkpoint_files:

    if (
        path.exists()
        and
        path.name
        ==
        "last_checkpoint.pth"
    ):

        deleted_bytes += (
            path.stat().st_size
        )


        path.unlink()


        deleted_files += 1


print(
    "Deleted last checkpoints:",
    deleted_files
)


print(
    "Recovered approximately:",
    round(
        deleted_bytes
        /
        (1024 ** 3),
        2
    ),
    "GB"
)


print(
    "\n✓ All best_model.pth files remain untouched."
)

In [ ]:
# ============================================
# Storage Cleanup 4
# Verify Important Phase 6 Models Remain
# ============================================

remaining_last = list(

    PHASE6_PATH.rglob(
        "last_checkpoint.pth"
    )
)


remaining_best = list(

    PHASE6_PATH.rglob(
        "best_model.pth"
    )
)


print(
    "Remaining last_checkpoint files:",
    len(
        remaining_last
    )
)


print(
    "Remaining best_model files:",
    len(
        remaining_best
    )
)


assert (
    len(
        remaining_last
    )
    ==
    0
)


assert (
    len(
        remaining_best
    )
    >=
    19
)


print(
    "\n✓ Cleanup verified."
)

print(
    "✓ Scientific best-model checkpoints preserved."
)

In [ ]:
# ============================================
# Section 13.2
# Prepare Standardized Visual Audit Sheet
# ============================================

phase8_visual_review_df = (

    phase8_selected_cases_df[
        [
            "Case_ID",
            "Domain",
            "Class",
            "Case_Type",
            "Fitzpatrick",
            "Correct",
            "Ensemble_Entropy"
        ]
    ]

    .copy()
)


phase8_visual_review_df[
    "Attention_Category"
] = ""


phase8_visual_review_df[
    "Primary_Attention_Region"
] = ""


phase8_visual_review_df[
    "Visible_Artifact_Attention"
] = ""


phase8_visual_review_df[
    "Reviewer_Confidence"
] = ""


phase8_visual_review_df[
    "Notes"
] = ""


PHASE8_VISUAL_REVIEW_FILE = (

    PHASE8_TABLE_PATH
    / "Phase8_Standardized_Visual_Audit.csv"
)


phase8_visual_review_df.to_csv(

    PHASE8_VISUAL_REVIEW_FILE,

    index=False
)


display(
    phase8_visual_review_df
)


print(
    "✓ Standardized visual-audit sheet created."
)

In [ ]:
# ============================================
# Section 13.3
# Standardized Qualitative Grad-CAM++ Visual Audit
# ============================================

visual_audit_coding = {

    "XAI_01": {
        "Attention_Category":
            "Mixed lesion and surrounding skin",
        "Primary_Attention_Region":
            "Central lesion-region and adjacent skin",
        "Visible_Artifact_Attention":
            "Unclear",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Activation broadly covers the central lesion-region and surrounding tissue."
    },

    "XAI_02": {
        "Attention_Category":
            "Mixed lesion and surrounding skin",
        "Primary_Attention_Region":
            "Central lower-leg/ankle region",
        "Visible_Artifact_Attention":
            "Yes",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Activation overlaps the clinical region but visible markings are also present."
    },

    "XAI_03": {
        "Attention_Category":
            "Predominantly lesion-focused",
        "Primary_Attention_Region":
            "Central focal lesion",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "High",
        "Notes":
            "Strong compact activation centered on the visible focal lesion."
    },

    "XAI_04": {
        "Attention_Category":
            "Predominantly lesion-focused",
        "Primary_Attention_Region":
            "Central focal lesion",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "High",
        "Notes":
            "Strong activation is concentrated around the visible lesion."
    },

    "XAI_05": {
        "Attention_Category":
            "Diffuse / non-localized",
        "Primary_Attention_Region":
            "Broad central and left skin region",
        "Visible_Artifact_Attention":
            "Unclear",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Attention is spatially broad without a clearly isolated focal region."
    },

    "XAI_06": {
        "Attention_Category":
            "Mixed lesion and surrounding skin",
        "Primary_Attention_Region":
            "Central-lower skin region",
        "Visible_Artifact_Attention":
            "Unclear",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Activation includes the apparent lesion-region together with surrounding skin."
    },

    "XAI_07": {
        "Attention_Category":
            "Predominantly background/artifact-focused",
        "Primary_Attention_Region":
            "Eyes and central facial anatomy",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "High",
        "Notes":
            "Attention is dominated by broader facial anatomy rather than a clearly isolated lesion."
    },

    "XAI_08": {
        "Attention_Category":
            "Predominantly lesion-focused",
        "Primary_Attention_Region":
            "Central nasal lesion region",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "High",
        "Notes":
            "Activation is strongly concentrated on the visible focal lesion."
    },

    "XAI_09": {
        "Attention_Category":
            "Predominantly lesion-focused",
        "Primary_Attention_Region":
            "Central focal lesion",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "High",
        "Notes":
            "Strong focal attention corresponds visually to the lesion-region."
    },

    "XAI_10": {
        "Attention_Category":
            "Predominantly lesion-focused",
        "Primary_Attention_Region":
            "Left focal lesion region",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Highest activation is concentrated around one visible focal lesion-region."
    },

    "XAI_11": {
        "Attention_Category":
            "Mixed lesion and surrounding skin",
        "Primary_Attention_Region":
            "Central lesion and adjacent skin",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Attention covers the lesion-region but extends substantially into adjacent tissue."
    },

    "XAI_12": {
        "Attention_Category":
            "Mixed lesion and surrounding skin",
        "Primary_Attention_Region":
            "Central oral lesion/anatomical region",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Activation includes the focal abnormal region together with neighboring oral anatomy."
    },

    "XAI_13": {
        "Attention_Category":
            "Predominantly lesion-focused",
        "Primary_Attention_Region":
            "Central focal lesion",
        "Visible_Artifact_Attention":
            "No",
        "Reviewer_Confidence":
            "High",
        "Notes":
            "Activation is predominantly concentrated over the visible focal lesion."
    },

    "XAI_14": {
        "Attention_Category":
            "Mixed lesion and surrounding skin",
        "Primary_Attention_Region":
            "Lesion-adjacent lower-left region",
        "Visible_Artifact_Attention":
            "Unclear",
        "Reviewer_Confidence":
            "Moderate",
        "Notes":
            "Strong activation extends beyond the focal region into nearby tissue."
    }
}


for case_id, coding in (
    visual_audit_coding.items()
):

    mask = (

        phase8_visual_review_df[
            "Case_ID"
        ]
        ==
        case_id
    )


    for column, value in (
        coding.items()
    ):

        phase8_visual_review_df.loc[
            mask,
            column
        ] = value


display(
    phase8_visual_review_df
)

In [ ]:
# ============================================
# Section 13.4
# Save Completed Visual Audit
# ============================================

phase8_visual_review_df.to_csv(

    PHASE8_VISUAL_REVIEW_FILE,

    index=False
)


assert (

    phase8_visual_review_df[
        "Attention_Category"
    ]
    .ne("")
    .all()
)


assert (

    phase8_visual_review_df[
        "Reviewer_Confidence"
    ]
    .ne("")
    .all()
)


print(
    "✓ All 14 frozen cases visually audited."
)

print(
    "✓ Completed visual audit saved:"
)

print(
    PHASE8_VISUAL_REVIEW_FILE
)

In [ ]:
# ============================================
# Section 13.5
# Visual Audit Summary
# ============================================

attention_summary = (

    phase8_visual_review_df[
        "Attention_Category"
    ]

    .value_counts()

    .rename_axis(
        "Attention_Category"
    )

    .reset_index(
        name="N"
    )
)


attention_summary[
    "Percent"
] = (

    attention_summary[
        "N"
    ]

    /
    len(
        phase8_visual_review_df
    )

    *
    100.0
)


display(
    attention_summary
)


print(
    "\nCorrect vs Attention Category:"
)


correct_attention_table = pd.crosstab(

    phase8_visual_review_df[
        "Correct"
    ],

    phase8_visual_review_df[
        "Attention_Category"
    ]
)


display(
    correct_attention_table
)


print(
    "\nFair/Dark vs Attention Category:"
)


domain_attention_table = pd.crosstab(

    phase8_visual_review_df[
        "Domain"
    ],

    phase8_visual_review_df[
        "Attention_Category"
    ]
)


display(
    domain_attention_table
)

In [ ]:
# ============================================
# Section 14.1
# Final Phase 8 XAI Summary
# ============================================

phase8_final_summary = {

    "Frozen_XAI_Cases":
        int(
            len(
                phase8_selected_cases_df
            )
        ),

    "Predominantly_Lesion_Focused":
        int(
            (
                phase8_visual_review_df[
                    "Attention_Category"
                ]
                ==
                "Predominantly lesion-focused"
            )
            .sum()
        ),

    "Mixed_Lesion_Surrounding":
        int(
            (
                phase8_visual_review_df[
                    "Attention_Category"
                ]
                ==
                "Mixed lesion and surrounding skin"
            )
            .sum()
        ),

    "Diffuse_Nonlocalized":
        int(
            (
                phase8_visual_review_df[
                    "Attention_Category"
                ]
                ==
                "Diffuse / non-localized"
            )
            .sum()
        ),

    "Background_Artifact_Focused":
        int(
            (
                phase8_visual_review_df[
                    "Attention_Category"
                ]
                ==
                "Predominantly background/artifact-focused"
            )
            .sum()
        )
}


phase8_final_summary[
    "Lesion_or_Mixed_N"
] = (

    phase8_final_summary[
        "Predominantly_Lesion_Focused"
    ]

    +

    phase8_final_summary[
        "Mixed_Lesion_Surrounding"
    ]
)


phase8_final_summary[
    "Lesion_or_Mixed_Percent"
] = (

    phase8_final_summary[
        "Lesion_or_Mixed_N"
    ]

    /

    phase8_final_summary[
        "Frozen_XAI_Cases"
    ]

    *

    100.0
)


for key, value in (
    phase8_final_summary.items()
):

    print(
        f"{key}: {value}"
    )

In [ ]:
# ============================================
# Section 14.2
# Create Phase 8 Completion Manifest
# ============================================

phase8_completion_manifest = {

    "Phase":
        "Phase 8",

    "Title":
        "Grad-CAM++ Explainability Analysis",

    "Authoritative_Model":
        "Phase 6 selected lambda=0",

    "Selected_Lambda":
        0.0,

    "Model_Seeds":
        [
            42,
            43,
            44,
            45,
            46
        ],

    "Explanation_Method":
        "Grad-CAM++",

    "Explanation_Aggregation":
        (
            "Five independently trained seed-specific "
            "Grad-CAM++ maps averaged per image"
        ),

    "Frozen_XAI_Cases":
        14,

    "Case_Selection_Used_Heatmaps":
        False,

    "Model_Selection_Reopened":
        False,

    "Numerical_Heatmaps_Saved":
        True,

    "Spatial_Stability_Analysis_Performed":
        True,

    "Exploratory_Uncertainty_Stability_Associations":
        True,

    "Entropy_vs_CAM_Correlation_Spearman_Rho":
        -0.261538,

    "Entropy_vs_CAM_Correlation_P":
        0.366411,

    "ProbabilitySD_vs_CAM_Correlation_Spearman_Rho":
        -0.279121,

    "ProbabilitySD_vs_CAM_Correlation_P":
        0.333845,

    "Entropy_vs_CAM_SD_Spearman_Rho":
        0.367033,

    "Entropy_vs_CAM_SD_P":
        0.196739,

    "Predominantly_Lesion_Focused_N":
        6,

    "Mixed_Lesion_Surrounding_N":
        6,

    "Diffuse_Nonlocalized_N":
        1,

    "Background_Artifact_Focused_N":
        1,

    "Lesion_or_Mixed_N":
        12,

    "Lesion_or_Mixed_Percent":
        85.714286,

    "Correct_Case_CAM_Correlation_Mean":
        0.578486,

    "Incorrect_Case_CAM_Correlation_Mean":
        0.644090,

    "Primary_Interpretation":
        (
            "Across the 14 frozen cases, Grad-CAM++ "
            "attention was predominantly lesion-focused "
            "or jointly distributed across lesion and "
            "surrounding skin in 85.7% of cases."
        ),

    "Correctness_Interpretation":
        (
            "Lesion-focused attention occurred in both "
            "correct and incorrect predictions, indicating "
            "that spatial localization alone did not "
            "guarantee correct classification."
        ),

    "Stability_Interpretation":
        (
            "Prediction uncertainty showed descriptive "
            "tendencies toward less stable explanations, "
            "but the exploratory Spearman associations "
            "were not statistically supported."
        ),

    "Important_Limitation":
        (
            "No lesion segmentation masks were available; "
            "therefore Grad-CAM++ localization was assessed "
            "using mask-free spatial metrics and structured "
            "visual review rather than verified lesion-overlap metrics."
        ),

    "Display_Modification":
        (
            "Viewer-friendly figures may obscure or soften "
            "clinical imagery without altering frozen cases "
            "or numerical Grad-CAM++ maps."
        ),

    "Status":
        "COMPLETE",

    "Frozen":
        True
}


PHASE8_COMPLETION_FILE = (

    PHASE8_MANIFEST_PATH
    / "Phase8_Completion_Manifest.json"
)


with open(
    PHASE8_COMPLETION_FILE,
    "w"
) as f:

    json.dump(
        phase8_completion_manifest,
        f,
        indent=4
    )


print(
    "✓ Phase 8 completion manifest saved:"
)

print(
    PHASE8_COMPLETION_FILE
)

In [ ]:
# ============================================
# Section 14.3
# Final Phase 8 Hard Completion Gate
# ============================================

assert (
    PHASE8_COMPLETION_FILE.exists()
)


with open(
    PHASE8_COMPLETION_FILE,
    "r"
) as f:

    final_phase8_manifest = json.load(
        f
    )


assert (
    final_phase8_manifest[
        "Status"
    ]
    ==
    "COMPLETE"
)


assert (
    final_phase8_manifest[
        "Frozen"
    ]
    is True
)


assert (
    final_phase8_manifest[
        "Selected_Lambda"
    ]
    ==
    0.0
)


assert (
    final_phase8_manifest[
        "Frozen_XAI_Cases"
    ]
    ==
    14
)


assert (
    final_phase8_manifest[
        "Case_Selection_Used_Heatmaps"
    ]
    is False
)


assert (
    final_phase8_manifest[
        "Model_Selection_Reopened"
    ]
    is False
)


assert (
    final_phase8_manifest[
        "Lesion_or_Mixed_N"
    ]
    ==
    12
)


print(
    "✓ Phase 8 status: COMPLETE"
)

print(
    "✓ Phase 8 frozen: TRUE"
)

print(
    "✓ 14 frozen XAI cases retained."
)

print(
    "✓ Five-seed ensemble Grad-CAM++ completed."
)

print(
    "✓ Numerical and qualitative XAI analyses completed."
)

print(
    "✓ No model selection was reopened."
)

print(
    "✓ No further Phase 8 case replacement is permitted."
)